# Hackathon ONE - Proyecto G9 | Eficiencia Energética Inteligente
## Sección 1: Introducción, Origen de los Datos y Mapeo de Variables

### 1.1. Origen de los Datos (Dataset de Referencia)
Para la construcción de este modelo predictivo de eficiencia energética, se utilizó el dataset público **RECS 2009 (Residential Energy Consumption Survey)**, recopilado por la *U.S. Energy Information Administration (EIA)*.

**Justificación técnica de la fuente:**
* **Volumen y Diversidad:** Contiene más de 12,000 registros reales detallados sobre características de viviendas, equipamiento electrodoméstico y consumos energéticos facturados de manera real.
* **Consistencia Física:** Evita los sesgos de distribuciones sintéticas o simuladas al azar que carecen de correlación termodinámica real.

---

### 1.2. Mapeo e Ingeniería de Características (Feature Engineering)
Para adaptarnos estrictamente al Producto Mínimo Viable (MVP) definido en los requerimientos del cliente, seleccionamos, limpiamos y transformamos las variables de RECS de la siguiente manera:

1. **`consumo_kwh` (Consumo Mensual Promedio):** * *Origen:* Variable `KWH` (Consumo anual total de electricidad en la encuesta).
   * *Transformación:* $consumo\_kwh = \frac{KWH}{12}$ (Normalizado a escala mensual para coincidir con la visualización típica de facturas de los usuarios).

2. **`tipo_inmueble` (Estructura de la Vivienda):**
   * *Origen:* Variable `TYPEHUQ` (Tipo de unidad de vivienda).
   * *Transformación:* Mapeo de categorías unifamiliares (Casas independientes, adosadas, etc.) a la categoría `"Casa"`, y departamentos de edificios a la categoría `"Departamento"`.

3. **`cantidad_equipos` (Carga Tecnológica del Hogar):**
   * *Origen:* Sumatoria de variables de inventario electrodoméstico (`TVCOLOR`, `NUMPC`, `FREEZER`, `STOVE`, `OVEN`, `MICRO`).
   * *Transformación:* Se sumaron los equipos activos de alto consumo, añadiendo una base fija de $+2$ unidades para representar el consumo básico pasivo de iluminación y climatización estándar.

4. **`horas_alto_consumo` (Persistencia de la Demanda):**
   * *Origen/Diseño:* Variable simulada proporcionalmente mediante el consumo mensual y la densidad de electrodomésticos, añadiendo ruido Gaussiano para mantener una distribución realista entre $1.0$ y $16.0$ horas diarias.

5. **`uso_horario_pico` (Hábitos de Consumo):**
   * *Origen/Diseño:* Variable booleana simulada probabilísticamente. El consumo elevado aumenta la probabilidad de consumo concurrente en horas de alta demanda en la red eléctrica.

---

### 1.3. Criterio de Negocio: Definición del Perfil de Eficiencia (Target)
La variable objetivo (`categoria`) clasifica a las viviendas en tres perfiles operativos (**Eficiente**, **Moderado**, **Ineficiente**). Esta clasificación se construyó a través de un sistema de puntuación lógico que penaliza la ineficiencia:

* **Consumo Base (Puntaje: 1 a 5):** Evaluado según el umbral de consumo mensual ($<350\text{ kWh}$, $<800\text{ kWh}$ o superior).
* **Demanda Crítica (Puntaje: 1 a 5):** Evaluado según las horas diarias de alto consumo diario.
* **Penalización por Hábitos (Puntaje: +2):** Si el usuario utiliza la energía predominantemente en horas pico (`uso_horario_pico = True`).

**Matriz de Decisión:**
* $Score \le 4$: **Eficiente** (Consumo controlado, bajo uso intensivo, hábitos fuera de pico).
* $5 \le Score \le 8$: **Moderado** (Consumo balanceado).
* $Score \ge 9$: **Ineficiente** (Altos consumos, uso simultáneo prolongado y/o hábitos concurrentes en hora pico).

### Celda 1: Carga Dinámica e Inspección de la Estructura de Datos

Esta celda tiene como objetivo principal importar las librerías base y cargar nuestro set de datos procesado (`data_procesada_hackaton.csv`). Está diseñada con un mecanismo de contingencia para asegurar que cualquier miembro del equipo pueda ejecutar el flujo de trabajo, sin importar si el repositorio es público o privado.

#### Explicación de las funciones del código:
1. **`import pandas as pd` e `import os`**:
   * `pandas` es la librería estándar de Python utilizada para la manipulación y análisis de estructuras de datos (DataFrames).
   * `os` nos permite interactuar con el sistema operativo de la máquina virtual de Google Colab (por ejemplo, para verificar si un archivo existe físicamente en el entorno).
2. **`os.path.exists(archivo_local)`**:
   * Evalúa si el archivo CSV ya fue subido manualmente a la barra lateral de Colab. Si es así, prioriza la carga local para evitar demoras por latencia de red.
3. **`pd.read_csv(...)`**:
   * Es la función encargada de parsear el archivo separado por comas (CSV) y transformarlo en un objeto DataFrame bidimensional en memoria.
4. **Estructura de Control `try-except`**:
   * Funciona como un sistema de protección. Si la carga local falla y la URL de GitHub arroja un error (por ejemplo, si el repositorio pasa a ser privado), atrapa la excepción (`Exception`) y le muestra al usuario instrucciones claras en pantalla de cómo solucionarlo en lugar de romper la ejecución.
5. **`df.shape`**:
   * Atributo que devuelve una tupla con la dimensionalidad del dataset `(filas, columnas)`. Nos sirve para validar que no se hayan perdido registros durante la carga.
6. **`df.info()`**:
   * Método de Pandas que imprime un resumen conciso del DataFrame. Muestra el número total de columnas, los nombres de cada una, el recuento de valores no nulos (útil para verificar que no haya datos faltantes) y los tipos de datos lógicos asignados (`float64`, `int64`, `bool`, `object`).

In [4]:
import pandas as pd

# Carga de datos desde GitHub
url = 'https://raw.githubusercontent.com/No-Country-simulation/Hackaton_G9_Server_2/refs/heads/feature/generacion-datos/data/data_procesada_hackaton.csv'
df = pd.read_csv(url)

# Inspección básica del Dataset
print("=== ESTRUCTURA DEL DATASET ===")
print(f"Número de Registros (Filas): {df.shape[0]}")
print(f"Número de Atributos (Columnas): {df.shape[1]}\n")

print("=== TIPOS DE DATOS Y VALORES NULOS ===")
print(df.info())

=== ESTRUCTURA DEL DATASET ===
Número de Registros (Filas): 12083
Número de Atributos (Columnas): 6

=== TIPOS DE DATOS Y VALORES NULOS ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12083 entries, 0 to 12082
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   consumo_kwh         12083 non-null  float64
 1   uso_horario_pico    12083 non-null  bool   
 2   cantidad_equipos    12083 non-null  int64  
 3   tipo_inmueble       12083 non-null  object 
 4   horas_alto_consumo  12083 non-null  float64
 5   categoria           12083 non-null  object 
dtypes: bool(1), float64(2), int64(1), object(2)
memory usage: 483.9+ KB
None


### Celda 2: Análisis Estadístico Descriptivo y de Proporciones

Una vez que comprobamos que la estructura del dataset es correcta, procedemos a realizar un análisis estadístico avanzado.

#### Explicación de las funciones del código:
1. **`df[[...]].describe().T`**:
   * `.describe()` calcula automáticamente las estadísticas descriptivas básicas para las columnas numéricas especificadas: media (`mean`), desviación estándar (`std` - qué tan dispersos están los datos respecto al promedio), valores mínimos (`min`) y máximos (`max`), y los percentiles (25%, 50%, 75%).
   * El sufijo `.T` transpone la tabla resultante (convierte filas en columnas) para que sea mucho más cómoda de leer en pantalla.
2. **`df[[...]].median()`**:
   * Calcula la mediana (el valor central exacto de la muestra ordenada). Comparar la mediana con la media es el primer paso para detectar asimetrías.
3. **`df[[...]].skew()`**:
   * Calcula el coeficiente de asimetría (skewness) de Fisher. Un valor mayor a 1 indica una asimetría positiva fuerte (cola larga a la derecha, como nuestro consumo de energía), lo que nos avisa que ciertos modelos lineales necesitarán una transformación de escala en las próximas semanas.
4. **`df['tipo_inmueble'].value_counts(normalize=True) * 100`**:
   * `value_counts()` cuenta cuántos hogares pertenecen a cada categoría (`Casa` o `Departamento`).
   * Al pasarle el parámetro `normalize=True` y multiplicarlo por 100, convierte las frecuencias absolutas en porcentajes de participación sobre el total, permitiéndonos analizar el balance del perfil habitacional.
5. **Bucle `for idx in freq_pico.index:`**:
   * Es una estructura cíclica que recorre cada categoría lógica de la variable booleana `uso_horario_pico`.
   * Dentro del bucle, usamos un condicional de Python para traducir el valor lógico (`True` / `False`) a un lenguaje natural legible para el equipo y el negocio: *"Sí consume en pico"* o *"No consume en pico"*, imprimiendo de manera prolija la distribución exacta de los hábitos de consumo.

In [6]:
import pandas as pd
import numpy as np

print("Analisis Estadistico Descriptivo Avanzado")
# 1. Estadísticas para variables numéricas continuas y discretas
print("--- Medidas de Tendencia Central y Dispersión ---")
stats_numericas = df[['consumo_kwh', 'cantidad_equipos', 'horas_alto_consumo']].describe().T
# Añadimos la mediana (50%) explícitamente y la asimetría para ver la forma de la distribución
stats_numericas['median'] = df[['consumo_kwh', 'cantidad_equipos', 'horas_alto_consumo']].median()
stats_numericas['skewness (asimetría)'] = df[['consumo_kwh', 'cantidad_equipos', 'horas_alto_consumo']].skew()
print(stats_numericas[['mean', 'median', 'std', 'min', 'max', 'skewness (asimetría)']].round(2))
print("\n" + "-"*66 + "\n")

# 2. Análisis de proporciones para variables categóricas y booleanas
print("--- Frecuencias y Proporciones: tipo_inmueble ---")
prop_inmueble = df['tipo_inmueble'].value_counts(normalize=True) * 100
freq_inmueble = df['tipo_inmueble'].value_counts()
for idx in freq_inmueble.index:
    print(f"  * {idx}: {freq_inmueble[idx]} hogares ({prop_inmueble[idx]:.2f}%)")

print("\n--- Frecuencias y Proporciones: uso_horario_pico ---")
prop_pico = df['uso_horario_pico'].value_counts(normalize=True) * 100
freq_pico = df['uso_horario_pico'].value_counts()
for idx in freq_pico.index:
    estado = "Sí consume en pico" if idx else "No consume en pico"
    print(f"  * {estado}: {freq_pico[idx]} hogares ({prop_pico[idx]:.2f}%)")

Analisis Estadistico Descriptivo Avanzado
--- Medidas de Tendencia Central y Dispersión ---
                      mean  median     std   min       max  \
consumo_kwh         940.68  801.92  636.77  1.42  12521.17   
cantidad_equipos      7.64    7.00    2.59  2.00     26.00   
horas_alto_consumo    8.28    7.70    3.92  1.00     16.00   

                    skewness (asimetría)  
consumo_kwh                         2.11  
cantidad_equipos                    0.95  
horas_alto_consumo                  0.44  

------------------------------------------------------------------

--- Frecuencias y Proporciones: tipo_inmueble ---
  * Casa: 9234 hogares (76.42%)
  * Departamento: 2849 hogares (23.58%)

--- Frecuencias y Proporciones: uso_horario_pico ---
  * Sí consume en pico: 6329 hogares (52.38%)
  * No consume en pico: 5754 hogares (47.62%)
